# 2. Neural Networks & Deep Learning
You should be familiar with how neural networks work in general, since that wont be covered in this notebook. We will go over how and when to use them. We will not look into Computer vision and NLP id est we will only look at multi layer perceptrons, not recurrent or convolutional networks.

installing cuda version of torch to use GPU

## MLP
A multi layer perceptron is a feed forward neural network. It consists of an input layer, one or more hidden layers and an output layer. Each layer consists of one or more neurons. Each neuron in a layer is connected to every neuron in the next layer. The connections between neurons are called weights. The weights are learned during training. The output of each neuron is computed as a weighted sum of its inputs, passed through an activation function. The activation function introduces non-linearity into the network, allowing it to learn complex functions(several linear layers can be represented as a single linear layer). Depending on the task, the output layer will have a different activation function. in the following we show a table of common activation functions and their use cases.

| Activation Function | Use Case                                  | what is does |
|---------------------|-------------------------------------------|--------------|
| Sigmoid             | Binary classification                     | Maps any real number to a value between 0 and 1 |
| Tanh                | Hidden layers                             | Maps any real number to a value between -1 and 1 |
| ReLU                | Hidden layers                             | Returns the input if it is positive, otherwise returns 0 |
| Softmax             | Multi-class classification (output layer) | Converts a vector of values into a probability distribution |

### Errors
MLPs use backpropagation to learn the weights. The error is computed as the difference between the predicted output and the true output. The error is then propagated back through the network, and the weights are updated using gradient descent. The loss function used to compute the error depends on the task. in the following we show a table of common loss functions and their use cases.

| Loss Function       | Use Case       | What it does                                                                                                                   |
|---------------------|----------------|--------------------------------------------------------------------------------------------------------------------------------|
| Mean Squared Error  | Regression     | Computes the average of the squared differences between the predicted and true values                                          |
| Mean Absolute Error | Regression     | Computes the average of the absolute differences between the predicted and true values                                         |
| Cross-Entropy Loss  | Classification | Computes the loss for a classification task, measuring the difference between the predicted and true probability distributions |

Neual networks can be used for both classification and regression tasks. For regression tasks, the output layer will have a linear activation function. For classification tasks, the output layer will have a sigmoid or softmax activation function, depending on whether it is binary or multi-class classification. We test both on the same dasets as the other notebooks.

### Regression

#### Data loading

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine, load_diabetes
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt

wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)

diabetes = load_diabetes()
df_diabetes = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df_diabetes["target"] = diabetes.target

df_diabetes.head()

X, X_test, y, y_test = train_test_split(df_diabetes.drop(columns=["target"]), df_diabetes["target"], test_size=0.2, random_state=42, shuffle = True)

#### Regression with MLP
We implement an MLP using torch with two hidden layers, each with a ReLU activation function and an output layer. The output size will be one since we want a scalar value as the output. We will use the mean squared error loss function and the AdamW optimizer. We will train the model for 100 epochs and plot the loss curve.

##### Model definition

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class Net(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.drouput = nn.Dropout(0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.drouput(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.drouput(out)
        out = self.fc3(out)
        return out

##### Training

In [ ]:
from sklearn.metrics import r2_score

model = Net(input_size=X.shape[1], hidden_size1=64, hidden_size2=32, output_size=1)
optimizer = optim.AdamW(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# standardize the target so the loss scale is ~1 instead of ~30000; undo it for metrics
y_mean, y_std = y.mean(), y.std()
X_train_t = torch.tensor(X.values, dtype=torch.float32)
y_train_t = torch.tensor(((y - y_mean) / y_std).values, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test.values, dtype=torch.float32)

def predict(X_t):
    model.eval()
    with torch.no_grad():
        return model(X_t).squeeze(1).numpy() * y_std + y_mean

epochs = 500
losses = []
acc = []
acc_test = []

for epoch in range(epochs):
    model.train()
    pred = model(X_train_t)
    loss = criterion(pred, y_train_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    acc.append(r2_score(y, predict(X_train_t)))
    acc_test.append(r2_score(y_test, predict(X_test_t)))
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch}: loss {loss.item():.3f}, "
              f"train R2 {r2_score(y, predict(X_train_t)):.3f}, "
              f"test R2 {r2_score(y_test, predict(X_test_t)):.3f}")

fig, ax = plt.subplots(1, 3, figsize=(30, 5))

ax[0].plot(losses)
ax[1].plot(acc)
ax[2].plot(acc_test)
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("MSE loss (standardized target)")
ax[0].set_title("Training loss")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("R2 Score")
ax[1].set_title("Training R2")
ax[2].set_xlabel("Epoch")
ax[2].set_ylabel("R2 Score")
ax[2].set_title("Test R2")
plt.show()


we see slight overfitting, but the model is able to learn the data well. The R2 score on the test set is 0.55, which is not great, but it is better than a linear regression model.

### Classification

#### Data loading

In [ ]:
X, X_test, y, y_test = train_test_split(df_wine, wine.target, test_size=0.2, random_state=42, shuffle = True)

#### Model Definition

In [ ]:
class Net(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.drouput = nn.Dropout(0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size2, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.drouput(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.drouput(out)
        out = self.fc3(out)
        out = self.softmax(out)
        return out

#### Training

In [ ]:
from sklearn.metrics import accuracy_score

model = Net(input_size=X.shape[1], hidden_size1=64, hidden_size2=32, output_size=len(np.unique(y)))
optimizer = optim.AdamW(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

X_train_t = torch.tensor(X.values, dtype=torch.float32)
y_train_t = torch.tensor(y.values, dtype=torch.float32)
X_test_t = torch.tensor(X_test.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

epochs = 100

for epoch in range(epochs):
    pred = model(torch.tensor(X.values, dtype=torch.float32))
    loss = criterion(pred, torch.tensor(y.values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch}: loss {loss.item():.3f}, "
              f"train acc {accuracy_score(y, pred.argmax(dim=1)):.3f}, "
              f"test acc {accuracy_score(y_test, model(torch.tensor(X_test.values, dtype=torch.float32)).argmax(dim=1)):.3f}")


### Syllabus check
consider the following topics of the syllabus covered:

##### Neural Networks
- Perceptron Basics
- Gradient Descent
- Backpropagation
- Activation Functions (ReLU, Sigmoid, Tanh)
- Loss Functions (MSE, MAE, Cross Entropy, etc.)
##### Deep Learning
- [x] Multi-Layer Perceptrons (MLP)
- Data Embeddings (text, image, audio)
- [ ] Pooling Techniques (Max, Average)
- [ ] Attention Mechanism
- [ ] Transformers (theory needed only for text and image)
- [ ] Autoencoders
- [x] SGD, Mini-Batch Gradient Descent
- [x] Momentum Methods (Adam, AdamW)
- [x] Convergence and Learning Rates
- [x] Regularization: Dropout, Early Stopping, Weight Decay
- [ ] Weight Initialization
- [ ] Batch Normalization
- [ ] Model Finetuning (full and parameter-efficient)

the topics not covered from the DL section can not be covered without covering later topics in Computer vision and NLP, which are not covered in this notebook. The topics covered in the DL section are enough to understand how to use MLPs for regression and classification tasks.